In [1]:
# ===== Optimized Optimum Filter (V1.2-fast) =====
import numpy as np
from numba import njit, prange
from scipy.fft import fft, ifft

# --- numba kernels (SIMD-friendly; fused amp/chi0; no divides) ---

@njit(cache=True, fastmath=True, nogil=True, parallel=True)
def _amp_chi0_from_X(Fr, Fi, inv_S, Xr, Xi):
    """Compute real(dot(F, X)) and sum(|X|^2 * inv_S) without temporaries."""
    s_amp = 0.0
    s_ch0 = 0.0
    for m in prange(Xr.size):
        s_amp += Fr[m] * Xr[m] - Fi[m] * Xi[m]
        xr = Xr[m]
        xi = Xi[m]
        s_ch0 += (xr * xr + xi * xi) * inv_S[m]
    return s_amp, s_ch0


@njit(cache=True, fastmath=True, nogil=True, parallel=True)
def _slide_and_eval_opt(a_seq, b_seq, Fr, Fi, Er, Ei, inv_S, Xr, Xi,
                        scale_fs_over_N, kernel_norm, start, hop, steps,
                        amps, chisqs, out_offset):
    """
    Sliding DFT + evaluation for 'steps' windows.
    a_seq[t] = x[t]/fs, b_seq[t] = x[t+N]/fs (pre-divided).
    Xr/Xi updated in place; outputs written to amps/chisqs.
    """
    t0 = start - hop
    N = Xr.size

    for s in range(steps):
        # advance by 'hop' micro-steps
        for u in range(hop):
            t = t0 + u + s * hop
            a = a_seq[t]
            b = b_seq[t]
            # X <- (X - a + b) * E  (E has unit magnitude)
            for m in prange(N):
                xr = Xr[m] - a + b
                xi = Xi[m]
                er = Er[m]
                ei = Ei[m]
                nr = xr * er - xi * ei
                ni = xr * ei + xi * er
                Xr[m] = nr
                Xi[m] = ni

        # evaluate amp and chi0
        s_amp = 0.0
        s_ch0 = 0.0
        for m in prange(N):
            s_amp += Fr[m] * Xr[m] - Fi[m] * Xi[m]
            xr = Xr[m]
            xi = Xi[m]
            s_ch0 += (xr * xr + xi * xi) * inv_S[m]

        amp = s_amp * scale_fs_over_N
        chi0 = s_ch0 * scale_fs_over_N
        chisq = (chi0 - amp * amp * kernel_norm) / (N - 2.0)
        amps[out_offset + s] = amp
        chisqs[out_offset + s] = chisq


class OptimumFilterV12Fast:
    """
    Same interface/behavior as your OptimumFilter V1.2,
    but with faster sliding_fit (applies ideas 2, 6, 7).
    """

    def __init__(self, template, noise_psd, sampling_frequency):
        self._template = np.asarray(template, dtype=np.float64)
        self._noise_psd = np.asarray(noise_psd, dtype=np.float64)
        self._sampling_frequency = float(sampling_frequency)
        self._update_state()

    def set_template(self, template):
        self._template = np.asarray(template, dtype=np.float64)
        self._update_state()

    def set_noise_psd(self, noise_psd):
        self._noise_psd = np.asarray(noise_psd, dtype=np.float64)
        self._update_state()

    def _update_state(self):
        fs = self._sampling_frequency
        self._length = int(self._template.size)
        N = self._length

        # Unfold PSD (exactly like original), but also keep its inverse
        if N % 2 == 0:
            noise_unf = np.concatenate((
                [np.inf],
                self._noise_psd[1:-1] / 2.0,
                [self._noise_psd[-1]],
                self._noise_psd[-2:0:-1] / 2.0
            ))
        else:
            noise_unf = np.concatenate((
                [np.inf],
                self._noise_psd[1:] / 2.0,
                self._noise_psd[-1:0:-1] / 2.0
            ))
        self._noise_psd_unfolded = np.ascontiguousarray(noise_unf, dtype=np.float64)
        self._inv_noise_psd_unfolded = np.ascontiguousarray(1.0 / self._noise_psd_unfolded, dtype=np.float64)
        self._inv_noise_psd_unfolded[0] = 0.0  # 1/inf -> 0

        # FFTs and filter kernel (same math as original)
        self._template_fft = fft(self._template) / fs
        self._kernel_fft = self._template_fft.conjugate() / self._noise_psd_unfolded
        self._kernel_normalization = (
            np.real(np.dot(self._kernel_fft, self._template_fft)) * fs / N
        )
        self._filter_kernel = self._kernel_fft / self._kernel_normalization

        # Pre-split for SIMD: F, E real/imag parts (contiguous float64)
        F = np.ascontiguousarray(self._filter_kernel.astype(np.complex128))
        m = np.arange(N, dtype=np.float64)
        E = np.exp(2j * np.pi * m / N).astype(np.complex128)

        self._F_real = np.ascontiguousarray(F.real)
        self._F_imag = np.ascontiguousarray(F.imag)
        self._E_real = np.ascontiguousarray(E.real)
        self._E_imag = np.ascontiguousarray(E.imag)

    # --- Same public API as original ---

    def fit(self, trace):
        trace_fft = fft(trace, axis=-1) / self._sampling_frequency
        trace_filtered = self._filter_kernel * trace_fft
        amp = np.real(trace_filtered.sum(axis=-1)) * self._sampling_frequency / self._length
        chisq0 = np.real((trace_fft.conj() * trace_fft / self._noise_psd_unfolded).sum()) \
                 * self._sampling_frequency / self._length
        chisq = (chisq0 - amp ** 2 * self._kernel_normalization) / (self._length - 2)
        return float(amp), float(chisq)

    def fit_with_shift(self, trace, allowed_shift_range=None):
        trace_fft = fft(trace, axis=-1) / self._sampling_frequency
        trace_filtered = self._filter_kernel * trace_fft
        trace_filtered_td = np.real(ifft(trace_filtered, axis=-1)) * self._sampling_frequency

        chi0 = np.real((trace_fft.conj() * trace_fft / self._noise_psd_unfolded).sum()) \
               * self._sampling_frequency / self._length
        chit_withdelay = (trace_filtered_td ** 2) * self._kernel_normalization
        chi = chi0 - chit_withdelay

        if allowed_shift_range is None:
            ind = np.arange(len(chi))
        else:
            N = self._length
            ind = np.concatenate((
                np.arange(N + allowed_shift_range[0], N),
                np.arange(allowed_shift_range[1] + 1)
            ))

        best_ind = ind[np.argmin(chi[ind], axis=-1)]
        amp = trace_filtered_td[best_ind]
        chisq = chi[best_ind] / (self._length - 3)
        t0 = int(best_ind if best_ind < self._length // 2 else best_ind - self._length)
        return float(amp), float(chisq), t0

    def sliding_fit(self, trace_long, hop=1, reanchor_every=None):
        x = np.ascontiguousarray(np.asarray(trace_long, dtype=np.float64))
        L = x.size
        N = self._length
        fs = self._sampling_frequency
        if N <= 0 or L < N:
            raise ValueError("Trace shorter than window length or invalid N.")
        if hop <= 0:
            raise ValueError("hop must be a positive integer.")

        num_windows = 1 + (L - N) // hop
        amps = np.empty(num_windows, dtype=np.float64)
        chisqs = np.empty(num_windows, dtype=np.float64)

        # precompute streams (no divides inside kernel)
        a_seq = np.ascontiguousarray(x[:-N] / fs)  # x[t]/fs
        b_seq = np.ascontiguousarray(x[N:] / fs)   # x[t+N]/fs

        # local SIMD-friendly views
        Fr = self._F_real
        Fi = self._F_imag
        Er = self._E_real
        Ei = self._E_imag
        invS = self._inv_noise_psd_unfolded
        kern_norm = float(self._kernel_normalization)
        scale = fs / N

        # initial anchor
        X0 = fft(x[0:N]) / fs
        Xr = np.ascontiguousarray(X0.real)
        Xi = np.ascontiguousarray(X0.imag)

        s_amp0, s_ch00 = _amp_chi0_from_X(Fr, Fi, invS, Xr, Xi)
        amp0 = s_amp0 * scale
        chi00 = s_ch00 * scale
        chisq0 = (chi00 - amp0 * amp0 * kern_norm) / (N - 2.0)
        amps[0] = amp0
        chisqs[0] = chisq0

        made = 1
        start = hop

        # main loop with optional re-anchors
        while start <= L - N:
            if reanchor_every and (made % reanchor_every == 0):
                Xk = fft(x[start:start + N]) / fs
                Xr = np.ascontiguousarray(Xk.real)
                Xi = np.ascontiguousarray(Xk.imag)
                s_amp, s_ch0 = _amp_chi0_from_X(Fr, Fi, invS, Xr, Xi)
                amp = s_amp * scale
                chi0 = s_ch0 * scale
                chisq = (chi0 - amp * amp * kern_norm) / (N - 2.0)
                amps[made] = amp
                chisqs[made] = chisq
                made += 1
                start += hop
            else:
                # number of windows until next reanchor or end
                if reanchor_every:
                    steps_to_reanchor = reanchor_every - (made % reanchor_every)
                else:
                    steps_to_reanchor = (L - N - start) // hop + 1
                steps_to_end = (L - N - start) // hop + 1
                steps = steps_to_reanchor if steps_to_reanchor < steps_to_end else steps_to_end
                if steps <= 0:
                    break

                _slide_and_eval_opt(
                    a_seq, b_seq, Fr, Fi, Er, Ei, invS, Xr, Xi,
                    scale, kern_norm, start, hop, steps,
                    amps, chisqs, made
                )
                made += steps
                start += steps * hop

        return amps, chisqs


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [2]:
import os, sys, ctypes, glob

# This is your conda env prefix
prefix = sys.exec_prefix
libdir  = os.path.join(prefix, "lib")

# 1) NVRTC 12.0 from this env
nvrtc_path = os.path.join(libdir, "libnvrtc.so.12")
ctypes.CDLL(nvrtc_path, mode=ctypes.RTLD_GLOBAL)

# 2) NVVM from this env (CUDA 12.0's soname is typically libnvvm.so.4)
#    We just pick the one in the env's lib folder.
nvvm_candidates = sorted(glob.glob(os.path.join(libdir, "libnvvm.so*")))
assert nvvm_candidates, "No libnvvm.so* found in your conda env; install cuda-nvvm=12.0.*"
nvvm_path = nvvm_candidates[-1]
ctypes.CDLL(nvvm_path, mode=ctypes.RTLD_GLOBAL)

# -- sanity print --
print("Using NVRTC:", nvrtc_path)
print("Using NVVM :", nvvm_path)

# Show NVRTC version to confirm 12.0
maj = ctypes.c_int(); min_ = ctypes.c_int()
ctypes.CDLL(nvrtc_path).nvrtcVersion(ctypes.byref(maj), ctypes.byref(min_))
print("NVRTC version:", f"{maj.value}.{min_.value}")  # expect 12.0


AssertionError: No libnvvm.so* found in your conda env; install cuda-nvvm=12.0.*

In [2]:
# ====== Reference (your original) for testing ======
import numpy as np
from scipy.fft import fft, ifft
from numba import njit

@njit(cache=True)
def _compute_amp(F, X, fs, N):
    s = 0.0
    for k in range(N):
        s += F[k].real * X[k].real - F[k].imag * X[k].imag
    return s * fs / N

@njit(cache=True)
def _compute_chi0(X, S_unf, fs, N):
    s = 0.0
    for k in range(N):
        xr = X[k].real
        xi = X[k].imag
        s += (xr * xr + xi * xi) / S_unf[k]
    return s * fs / N

@njit(cache=True)
def _slide_and_eval_ref(x, fs, F, S_unf, E, X, start, hop, steps, N, kernel_norm, amps, chisqs, out_offset):
    t0 = start - hop
    for s in range(steps):
        for u in range(hop):
            t = t0 + u + s * hop
            a = x[t] / fs
            b = x[t + N] / fs
            for m in range(N):
                X[m] = (X[m] - a + b) * E[m]
        amp = _compute_amp(F, X, fs, N)
        chi0 = _compute_chi0(X, S_unf, fs, N)
        chisq = (chi0 - amp * amp * kernel_norm) / (N - 2)
        amps[out_offset + s] = amp
        chisqs[out_offset + s] = chisq

class OptimumFilterRef:
    def __init__(self, template, noise_psd, sampling_frequency):
        self._template = np.asarray(template, dtype=np.float64)
        self._noise_psd = np.asarray(noise_psd, dtype=np.float64)
        self._sampling_frequency = float(sampling_frequency)
        self._update_state()

    def set_template(self, template):
        self._template = np.asarray(template, dtype=np.float64)
        self._update_state()

    def set_noise_psd(self, noise_psd):
        self._noise_psd = np.asarray(noise_psd, dtype=np.float64)
        self._update_state()

    def _update_state(self):
        self._length = len(self._template)
        if self._length % 2 == 0:
            self._noise_psd_unfolded = np.concatenate((
                [np.inf], self._noise_psd[1:-1] / 2, [self._noise_psd[-1]],
                self._noise_psd[-2:0:-1] / 2))
        else:
            self._noise_psd_unfolded = np.concatenate((
                [np.inf], self._noise_psd[1:] / 2, self._noise_psd[-1:0:-1] / 2))
        self._template_fft = fft(self._template) / self._sampling_frequency
        self._kernel_fft = self._template_fft.conjugate() / self._noise_psd_unfolded
        self._kernel_normalization = np.real(np.dot(self._kernel_fft, self._template_fft)) \
                                     * self._sampling_frequency / self._length
        self._filter_kernel = self._kernel_fft / self._kernel_normalization

    def fit(self, trace):
        trace_fft = fft(trace, axis=-1) / self._sampling_frequency
        trace_filtered = self._filter_kernel * trace_fft
        amp = np.real(trace_filtered.sum(axis=-1)) * self._sampling_frequency / self._length
        chisq0 = np.real((trace_fft.conj() * trace_fft / self._noise_psd_unfolded).sum()) \
                 * self._sampling_frequency / self._length
        chisq = (chisq0 - amp ** 2 * self._kernel_normalization) / (self._length - 2)
        return float(amp), float(chisq)

    def fit_with_shift(self, trace, allowed_shift_range=None):
        trace_fft = fft(trace, axis=-1) / self._sampling_frequency
        trace_filtered = self._filter_kernel * trace_fft
        trace_filtered_td = np.real(ifft(trace_filtered, axis=-1)) * self._sampling_frequency
        chi0 = np.real((trace_fft.conj() * trace_fft / self._noise_psd_unfolded).sum()) \
               * self._sampling_frequency / self._length
        chit_withdelay = (trace_filtered_td ** 2) * self._kernel_normalization
        chi = chi0 - chit_withdelay
        if allowed_shift_range is None:
            ind = np.arange(len(chi))
        else:
            ind = np.concatenate((
                np.arange(self._length + allowed_shift_range[0], self._length),
                np.arange(allowed_shift_range[1] + 1)))
        best_ind = ind[np.argmin(chi[ind], axis=-1)]
        amp = trace_filtered_td[best_ind]
        chisq = chi[best_ind] / (self._length - 3)
        t0 = int(best_ind if best_ind < self._length // 2 else best_ind - self._length)
        return float(amp), float(chisq), t0

    def sliding_fit(self, trace_long, hop=1, reanchor_every=None):
        x = np.asarray(trace_long, dtype=np.float64)
        L = x.size
        N = int(self._length)
        fs = float(self._sampling_frequency)
        if N <= 0 or L < N:
            raise ValueError("Trace shorter than window length or invalid N.")
        if hop <= 0:
            raise ValueError("hop must be a positive integer.")
        num_windows = 1 + (L - N) // hop
        amps = np.empty(num_windows, dtype=np.float64)
        chisqs = np.empty(num_windows, dtype=np.float64)
        m = np.arange(N, dtype=np.float64)
        E = np.exp(2j * np.pi * m / N).astype(np.complex128)
        F = np.asarray(self._filter_kernel, dtype=np.complex128)
        S_unf = np.asarray(self._noise_psd_unfolded, dtype=np.float64)
        kern_norm = float(self._kernel_normalization)
        X = fft(x[0:N]) / fs
        X = np.ascontiguousarray(X.astype(np.complex128))
        amp0 = _compute_amp(F, X, fs, N)
        chi00 = _compute_chi0(X, S_unf, fs, N)
        chisq0 = (chi00 - amp0 * amp0 * kern_norm) / (N - 2)
        amps[0] = amp0
        chisqs[0] = chisq0
        made = 1
        start = hop
        while start <= L - N:
            if reanchor_every and (made % reanchor_every == 0):
                X[:] = fft(x[start:start + N]) / fs
                amp = _compute_amp(F, X, fs, N)
                chi0 = _compute_chi0(X, S_unf, fs, N)
                chisq = (chi0 - amp * amp * kern_norm) / (N - 2)
                amps[made] = amp
                chisqs[made] = chisq
                made += 1
                start += hop
            else:
                if reanchor_every:
                    steps_to_reanchor = reanchor_every - (made % reanchor_every)
                else:
                    steps_to_reanchor = (L - N - start) // hop + 1
                steps_to_end = (L - N - start) // hop + 1
                steps = min(steps_to_reanchor, steps_to_end)
                if steps <= 0:
                    break
                _slide_and_eval_ref(
                    x=x, fs=fs, F=F, S_unf=S_unf, E=E, X=X,
                    start=start, hop=hop, steps=steps, N=N,
                    kernel_norm=kern_norm, amps=amps, chisqs=chisqs, out_offset=made
                )
                made += steps
                start += steps * hop
        return amps, chisqs

# ====== Quick functional test ======
def _make_test_data(N=256, fs=1_000.0, L=4096, amp=3.0, pos=1500, noise_sigma=0.5, seed=0):
    rng = np.random.default_rng(seed)
    # template: short exponential
    t = np.arange(N, dtype=np.float64) / fs
    tau = 0.010
    template = np.exp(-t / tau)
    template /= np.sqrt(np.sum(template**2))  # normalize energy
    # flat PSD (rfft length N//2+1)
    noise_psd = np.ones(N // 2 + 1, dtype=np.float64)
    # trace with noise + injected pulse
    x = rng.normal(0.0, noise_sigma, size=L).astype(np.float64)
    if pos + N <= L:
        x[pos:pos + N] += amp * template
    return template, noise_psd, fs, x

def test_equivalence():
    template, noise_psd, fs, x = _make_test_data()
    N = template.size

    ref = OptimumFilterRef(template, noise_psd, fs)
    fast = OptimumFilterV12Fast(template, noise_psd, fs)

    # Single-window fit
    w0 = x[0:N]
    a_ref, c_ref = ref.fit(w0)
    a_fast, c_fast = fast.fit(w0)

    # Fit-with-shift
    aws_ref = ref.fit_with_shift(w0)
    aws_fast = fast.fit_with_shift(w0)

    # Sliding fit
    amps_ref, chi_ref = ref.sliding_fit(x, hop=1, reanchor_every=64)
    amps_fast, chi_fast = fast.sliding_fit(x, hop=1, reanchor_every=64)

    # Assertions / diagnostics
    ok1 = np.allclose(a_ref, a_fast, rtol=1e-12, atol=1e-12)
    ok2 = np.allclose(c_ref, c_fast, rtol=1e-12, atol=1e-12)
    ok3 = np.allclose(aws_ref[0], aws_fast[0], rtol=1e-12, atol=1e-12) and \
          np.allclose(aws_ref[1], aws_fast[1], rtol=1e-12, atol=1e-12) and \
          (aws_ref[2] == aws_fast[2])
    ok4 = np.allclose(amps_ref, amps_fast, rtol=1e-10, atol=1e-10)
    ok5 = np.allclose(chi_ref, chi_fast, rtol=1e-10, atol=1e-10)

    print("fit amp equal      :", ok1, " | max abs diff:", float(np.max(np.abs(a_ref - a_fast))))
    print("fit chisq equal    :", ok2, " | max abs diff:", float(np.max(np.abs(c_ref - c_fast))))
    print("fit_with_shift eq  :", ok3)
    print("sliding amps equal :", ok4, " | max abs diff:", float(np.max(np.abs(amps_ref - amps_fast))))
    print("sliding chi² equal :", ok5, " | max abs diff:", float(np.max(np.abs(chi_ref - chi_fast))))

if __name__ == "__main__":
    test_equivalence()


fit amp equal      : True  | max abs diff: 0.0
fit chisq equal    : True  | max abs diff: 0.0
fit_with_shift eq  : True
sliding amps equal : True  | max abs diff: 3.9968028886505635e-15
sliding chi² equal : True  | max abs diff: 8.673617379884035e-19
